## Spam

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [12]:


df = pd.read_csv("./Datasets/spam.csv", encoding="latin-1")

# keep only required columns
df = df[["v1", "v2"]]

df.columns = ["label", "message"]

print(df.shape)   # MUST be ~ (5572, 2)

(5572, 2)


In [13]:
df["label"] = df["label"].map({"ham": 0, "spam": 1})

In [22]:
X = df["message"]
y = df["label"]

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1,2),   # IMPORTANT 🔥
    max_features=5000
)

X = vectorizer.fit_transform(df["message"])

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # VERY IMPORTANT 🔥
)

In [25]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [26]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9748878923766816
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       966
           1       0.99      0.82      0.90       149

    accuracy                           0.97      1115
   macro avg       0.98      0.91      0.94      1115
weighted avg       0.98      0.97      0.97      1115



In [27]:

## test
msg = ["Congratulations! You won a free ticket"]

msg_vec = vectorizer.transform(msg)
print(model.predict(msg_vec))   # 1 = spam

[0]


In [28]:
print(df.shape)
print(df["label"].value_counts())

(5572, 2)
label
0    4825
1     747
Name: count, dtype: int64


## Movie Recommendation

In [29]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
df = pd.read_csv("./Datasets/movie_recommendation.csv")

df = df[["movie_id", "title", "cast", "crew"]]
df.dropna(inplace=True)

In [32]:
df

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."
...,...,...,...,...
4798,9367,El Mariachi,"[{""cast_id"": 1, ""character"": ""El Mariachi"", ""c...","[{""credit_id"": ""52fe44eec3a36847f80b280b"", ""de..."
4799,72766,Newlyweds,"[{""cast_id"": 1, ""character"": ""Buzzy"", ""credit_...","[{""credit_id"": ""52fe487dc3a368484e0fb013"", ""de..."
4800,231617,"Signed, Sealed, Delivered","[{""cast_id"": 8, ""character"": ""Oliver O\u2019To...","[{""credit_id"": ""52fe4df3c3a36847f8275ecf"", ""de..."
4801,126186,Shanghai Calling,"[{""cast_id"": 3, ""character"": ""Sam"", ""credit_id...","[{""credit_id"": ""52fe4ad9c3a368484e16a36b"", ""de..."


In [33]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i["name"])
    return L

df["cast"] = df["cast"].apply(convert)

In [34]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i["job"] == "Director":
            L.append(i["name"])
    return L

df["crew"] = df["crew"].apply(fetch_director)

In [35]:
df["tags"] = df["cast"] + df["crew"]

# convert list → string
df["tags"] = df["tags"].apply(lambda x: " ".join(x))

In [36]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(df["tags"]).toarray()

In [37]:
similarity = cosine_similarity(vectors)

In [38]:
def recommend(movie):
    index = df[df["title"] == movie].index[0]
    distances = similarity[index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movies_list:
        print(df.iloc[i[0]].title)

In [39]:
recommend("Avatar")

The Dark Knight
Bottle Rocket
The Dark Knight Rises
Step Up 3D
Star Trek Into Darkness
